<!-- NB_DOC_INTRO_V1 -->
# ML — Bench multi-modeles + CV + tuning (GridSearch, RandomSearch, Optuna)

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**LE notebook quand tu demarres un projet** : tester rapidement plusieurs algos + trouver les bons hyperparametres. Couvre **8 algos classif + 6 algos regression** avec une **fonction bench reutilisable**. Tuning via **GridSearchCV** (exhaustif), **RandomizedSearchCV** (rapide), **Optuna** (bayesien moderne).

## Plan

1. Setup + datasets
2. Bench classification (8 algos)
3. Bench regression (6 algos)
4. Cross-validation propre
5. GridSearchCV (exhaustif)
6. RandomizedSearchCV (rapide)
7. Optuna (bayesien)
8. Comparaison strategies tuning
9. Pieges
10. References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet, SGDClassifier, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
)
from sklearn.svm import SVC, SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, r2_score, root_mean_squared_error
import time
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Datasets

In [ ]:
# Classification (binaire, 1000 samples, 20 features)
X_clf, y_clf = make_classification(n_samples=1000, n_features=20, n_informative=15,
                                    n_redundant=2, n_classes=2, random_state=SEED)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_clf, y_clf, test_size=0.25, stratify=y_clf, random_state=SEED)

# Regression
X_reg, y_reg = make_regression(n_samples=1000, n_features=15, n_informative=10,
                                noise=15, random_state=SEED)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_reg, y_reg, test_size=0.25, random_state=SEED)

print(f"Classif : {Xc_tr.shape}, classes {np.bincount(yc_tr)}")
print(f"Regression : {Xr_tr.shape}")

## 2. Bench classification — 8 algos

In [ ]:
classifiers = {
    "LogReg":            LogisticRegression(max_iter=1000, random_state=SEED),
    "KNN":               KNeighborsClassifier(n_neighbors=5),
    "DecisionTree":      DecisionTreeClassifier(random_state=SEED),
    "RandomForest":      RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    "GradientBoosting":  GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    "AdaBoost":          AdaBoostClassifier(n_estimators=50, random_state=SEED),
    "SVC (RBF)":         Pipeline([("scale", StandardScaler()), ("svc", SVC(random_state=SEED))]),
    "MLP":               Pipeline([("scale", StandardScaler()), ("mlp", MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=SEED))]),
    "NaiveBayes":        GaussianNB(),
    "SGD":               Pipeline([("scale", StandardScaler()), ("sgd", SGDClassifier(loss="log_loss", random_state=SEED))]),
}

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for name, clf in classifiers.items():
    t0 = time.perf_counter()
    scores = cross_val_score(clf, Xc_tr, yc_tr, cv=cv, scoring="f1", n_jobs=-1)
    t = time.perf_counter() - t0
    # Eval final sur test
    clf.fit(Xc_tr, yc_tr)
    test_f1 = f1_score(yc_te, clf.predict(Xc_te))
    results.append({"model": name, "cv_mean_f1": scores.mean(), "cv_std": scores.std(),
                    "test_f1": test_f1, "fit_time_s": t})

df_clf = pd.DataFrame(results).sort_values("cv_mean_f1", ascending=False)
print(df_clf.round(4))

## 3. Bench regression — 6 algos

In [ ]:
regressors = {
    "LinearRegression":  Ridge(alpha=0.0),
    "Ridge (L2)":        Ridge(alpha=1.0, random_state=SEED),
    "Lasso (L1)":        Lasso(alpha=0.1, random_state=SEED),
    "ElasticNet":        ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=SEED),
    "KNN":               KNeighborsRegressor(n_neighbors=5),
    "RandomForest":      RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    "GradientBoosting":  GradientBoostingRegressor(n_estimators=100, random_state=SEED),
    "SVR (RBF)":         Pipeline([("scale", StandardScaler()), ("svr", SVR())]),
}

results = []
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
for name, reg in regressors.items():
    t0 = time.perf_counter()
    scores = cross_val_score(reg, Xr_tr, yr_tr, cv=cv, scoring="r2", n_jobs=-1)
    t = time.perf_counter() - t0
    reg.fit(Xr_tr, yr_tr)
    test_r2 = r2_score(yr_te, reg.predict(Xr_te))
    test_rmse = root_mean_squared_error(yr_te, reg.predict(Xr_te))
    results.append({"model": name, "cv_mean_r2": scores.mean(), "cv_std": scores.std(),
                    "test_r2": test_r2, "test_rmse": test_rmse, "fit_time_s": t})

df_reg = pd.DataFrame(results).sort_values("cv_mean_r2", ascending=False)
print(df_reg.round(4))

## 4. Tuning — GridSearchCV (exhaustif)

In [ ]:
# Espace de recherche pour RandomForest classif
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth":    [None, 10, 20],
    "min_samples_split": [2, 5, 10],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_grid, cv=3, scoring="f1", n_jobs=-1, verbose=0,
)
t0 = time.perf_counter()
grid.fit(Xc_tr, yc_tr)
t_grid = time.perf_counter() - t0

print(f"GridSearch : {len(grid.cv_results_['params'])} combinaisons en {t_grid:.2f}s")
print(f"Best params : {grid.best_params_}")
print(f"Best CV F1  : {grid.best_score_:.4f}")
print(f"Test F1     : {f1_score(yc_te, grid.predict(Xc_te)):.4f}")

## 5. RandomizedSearchCV (rapide pour grand espace)

In [ ]:
from scipy.stats import randint

param_dist = {
    "n_estimators": randint(50, 300),
    "max_depth":    [None, 5, 10, 20, 50],
    "min_samples_split": randint(2, 20),
    "max_features": ["sqrt", "log2", 0.5, 0.7],
}

rand = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,                # 20 essais aleatoires (vs grid potentiellement enorme)
    cv=3, scoring="f1", random_state=SEED, n_jobs=-1,
)
t0 = time.perf_counter()
rand.fit(Xc_tr, yc_tr)
t_rand = time.perf_counter() - t0

print(f"RandomSearch : 20 essais en {t_rand:.2f}s  (vs grid {t_grid:.2f}s)")
print(f"Best params : {rand.best_params_}")
print(f"Best CV F1  : {rand.best_score_:.4f}")

## 6. Optuna (bayesien, recommande)

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth":    trial.suggest_int("max_depth", 3, 50),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5]),
        }
        clf = RandomForestClassifier(**params, random_state=SEED, n_jobs=-1)
        return cross_val_score(clf, Xc_tr, yc_tr, cv=3, scoring="f1", n_jobs=-1).mean()

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=20, show_progress_bar=False)
    t_optuna = time.perf_counter() - t0

    print(f"Optuna : 20 trials en {t_optuna:.2f}s")
    print(f"Best params : {study.best_params}")
    print(f"Best CV F1  : {study.best_value:.4f}")
except ImportError:
    print("Optuna pas installe (uv add --group ml optuna)")

## 7. Comparaison strategies tuning

| Strategie | Cas d'usage | Speedup vs Grid |
|---|---|---|
| **GridSearchCV** | Petit espace (< 50 combinaisons), sur sur | 1× (baseline) |
| **RandomizedSearchCV** | Grand espace, exploration | 5-50× |
| **Optuna (TPE bayesien)** | Bonne qualite, parallelisable, pruning | 10-100× pour atteindre meme score |
| **Hyperopt** (similar to Optuna) | Idem, API legerement differente | Idem |
| **BayesSearchCV** (scikit-optimize) | Bayesien dans sklearn API | 5-20× |

## 8. Pieges courants

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Tuning sur le test set | Toujours sur le train (CV), eval finale UNE FOIS sur test |
| `cv=5` sans `shuffle=True` | Donnees triees → folds biaises (KFold(shuffle=True)) |
| Pas de `random_state` | Reproductibilite cassee |
| GridSearch avec 10 params × 10 valeurs (10 milliards) | Random ou Optuna |
| `n_jobs=-1` qui explose la RAM | Tester avec `n_jobs=1` d'abord sur gros dataset |
| Pipeline absent → leakage scaler | `Pipeline([("scale", StandardScaler()), ...])` |
| Reporter la **best CV score** comme generalisation | Toujours eval sur test hold-out apres |


## References

### Documentation
- sklearn model_selection : https://scikit-learn.org/stable/modules/model_selection.html
- Optuna : https://optuna.readthedocs.io/
- Hyperopt : http://hyperopt.github.io/hyperopt/

### Voir aussi
- [ML_Bagging_Boosting.ipynb](ML_Bagging_Boosting.ipynb)
- [ML_Optimisation_de_Modèles.ipynb](ML_Optimisation_de_Modèles.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
- [INRIA_SKLearn_MOOC.ipynb](INRIA_SKLearn_MOOC.ipynb)
